In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Loading our final csv file from BigQuery
data_path = '/Users/robert/Desktop/DCU/hospital-re-admission/data/processed/mimic_readmission_final.csv'
df = pd.read_csv(data_path)

# For the basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Checking readmission rate
print("Target Variable Distribution:")
print(df['readmitted_30d'].value_counts())
print(f"\nReadmission rate: {df['readmitted_30d'].mean():.2%}")

# Vis.
plt.figure(figsize=(6,4))
df['readmitted_30d'].value_counts().plot(kind='bar')
plt.title('Class Distribution: 30-Day Readmission')
plt.xlabel('Readmitted (0=No, 1=Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Temporal Split
# Convert admission time to datetime
df['admittime'] = pd.to_datetime(df['admittime'])

# Check date range
print(f"Date range: {df['admittime'].min()} to {df['admittime'].max()}")

# Sort by time
df = df.sort_values('admittime').reset_index(drop=True)

# Define split point (80/20 temporal split)
split_idx = int(len(df) * 0.8)
split_date = df.iloc[split_idx]['admittime']

print(f"\nSplit date: {split_date}")

# Create train/test sets
train_df = df[df['admittime'] < split_date].copy()
test_df = df[df['admittime'] >= split_date].copy()

print(f"\nTrain: {len(train_df)} admissions ({train_df['readmitted_30d'].mean():.2%} readmission)")
print(f"Test: {len(test_df)} admissions ({test_df['readmitted_30d'].mean():.2%} readmission)")

In [ ]:
# Save to processed folder
train_df.to_csv('../../data/processed/train_data.csv', index=False)
test_df.to_csv('../../data/processed/test_data.csv', index=False)

print("Train/test splits saved!")

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print("Missing Values Summary:")
print(missing_summary[missing_summary['Percentage'] > 0].head(10))

# **Summary & Next To Dos:**
## 1. Temporal Split 

- Train: 324,824 admissions (17.32% readmission)
- Test: 81,207 admissions (17.90% readmission)
- Class balance: Nearly identical between splits (~0.6% difference), like we needed
- Split ratio: 80/20 - as per the standard practice we implied
- Dates: 2105-2179 (train) vs 2179-2214 (test) - thus, proper temporal separation

That means no data leakage means we will train on past and test on future.


## 2. Handling Missing Values, High Missing (>20%)

- days_to_next_admission (44% - Expected): Now this only exists if patient was readmitted. This is the target leakage risk—drop this feature entirely.
- days_since_last_discharge (40% - Expected): First-time admissions have no prior. So we will keep it and impute with -1 or create "no_prior_admission" flag.
- Lab values (glucose, sodium, potassium, creatinine, WBC): 23-26% missing - Depicting clinical reality cuz not all patients get all the labs.

In [ ]:
#S6
# 1. DROP LEAKAGE FEATURES
# days_to_next_admission is only known AFTER readmission happens
# Including it would give model perfect future knowledge
if 'days_to_next_admission' in df.columns:
    df = df.drop(columns=['days_to_next_admission'])
    print("Dropped 'days_to_next_admission' (target leakage)")

# 2. HANDLE PRIOR ADMISSION FEATURE
# Missing = first-time patient (no prior discharge)
df['has_prior_admission'] = (~df['days_since_last_discharge'].isna()).astype(int)
df['days_since_last_discharge'] = df['days_since_last_discharge'].fillna(-1)
print("Encoded 'days_since_last_discharge' (-1 = no prior)")

# 3. LAB VALUE IMPUTATION
# Missing labs = test not ordered (clinical decision)
# Impute with median (not mean, robust to outliers)
lab_features = [
    'glucose_min', 'glucose_max',
    'sodium_min', 'sodium_max', 
    'potassium_min', 'potassium_max',
    'creatinine_max', 'wbc_max'
]

for col in lab_features:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed {col} with median: {median_val:.2f}")

# 4. VERIFY NO MISSING VALUES LEFT
print("\n" + "="*50)
print("Remaining Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Check if same patients appear in train and test
train_patients = set(train_df['subject_id'].unique())
test_patients = set(test_df['subject_id'].unique())
overlap = train_patients & test_patients

print(f"Train patients: {len(train_patients)}")
print(f"Test patients: {len(test_patients)}")
print(f"Overlapping patients: {len(overlap)}")
print(f"Overlap %: {len(overlap)/len(train_patients)*100:.2f}%")

In [ ]:
# Save cleaned versions
train_df.to_csv('../../data/processed/train_data_clean.csv', index=False)
test_df.to_csv('../../data/processed/test_data_clean.csv', index=False)
print("Cleaned train/test splits saved!")

In [ ]:
#C8
# FINAL MISSING VALUE CLEANUP

# 1. MARITAL STATUS (10,178 missing = 2.5%)
# Missing = Unknown/Not documented
df['marital_status'] = df['marital_status'].fillna('UNKNOWN')
print("Filled marital_status with 'UNKNOWN'")

# 2. LANGUAGE (555 missing = 0.1%)
# Missing = English (which is the most common in MIMIC)
df['language'] = df['language'].fillna('ENGLISH')
print("Filled language with 'ENGLISH'")

# 3. INSURANCE (4,602 missing = 1.1%)
# Missing = Unknown coverage
df['insurance'] = df['insurance'].fillna('UNKNOWN')
print("Filled insurance with 'UNKNOWN'")

# 4. ADMISSION LOCATION (1 missing)
# Single missing value - will use mode (most common)
df['admission_location'] = df['admission_location'].fillna(df['admission_location'].mode()[0])
print("Filled admission_location with mode")

# 5. DISCHARGE LOCATION (53,135 missing = 13%)
# Missing = Still in hospital / died / not documented
df['discharge_location'] = df['discharge_location'].fillna('UNKNOWN')
print("Filled discharge_location with 'UNKNOWN'")

# 6. LAB TEST COUNTS (62,672 missing = 15.4%)
# Missing = No labs ordered in first 24h
df['num_lab_tests_24h'] = df['num_lab_tests_24h'].fillna(0)
df['num_abnormal_labs'] = df['num_abnormal_labs'].fillna(0)
print("Filled lab test counts with 0 (no tests ordered)")

# 7. HEMOGLOBIN (91,815 missing = 22.6%)
# Clinical lab value - imputing with median
df['hemoglobin_min'] = df['hemoglobin_min'].fillna(df['hemoglobin_min'].median())
print(f"Imputed hemoglobin_min with median: {df['hemoglobin_min'].median():.2f}")

#Now verifying at once.
remaining_missing = df.isnull().sum()
if remaining_missing.sum() == 0:
    print("\n" + "="*32)
    print("NO MISSING VALUES ARE REMAINING!")
    print("="*32)
else:
    print("\nStill missing:")
    print(remaining_missing[remaining_missing > 0])

In [ ]:
#C9
# RE-CREATING THE TRAIN/TEST SPLITS (Now that its clean)

# Sorting by time again (after all cleaning)
df = df.sort_values('admittime').reset_index(drop=True)

# Same splitting point as before
split_idx = int(len(df) * 0.8)
split_date = df.iloc[split_idx]['admittime']

# Creating final clean splits
train_df = df[df['admittime'] < split_date].copy()
test_df = df[df['admittime'] >= split_date].copy()

print(f"Final Clean Splits:")
print(f"Train: {len(train_df)} admissions ({train_df['readmitted_30d'].mean():.2%} readmission)")
print(f"Test: {len(test_df)} admissions ({test_df['readmitted_30d'].mean():.2%} readmission)")

# Saving to disk
train_df.to_csv('../../data/processed/train_data_clean.csv', index=False)
test_df.to_csv('../../data/processed/test_data_clean.csv', index=False)

print("\nCleaned train/test splits saved!")
print("   - ../../data/processed/train_data_clean.csv")
print("   - ../../data/processed/test_data_clean.csv")

In [ ]:
# ============================================
# DATASET SUMMARY - COPY THIS TO YOUR REPORT
# ============================================

print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"Total admissions: {len(df):,}")
print(f"Unique patients: {df['subject_id'].nunique():,}")
print(f"Date range: {df['admittime'].min()} to {df['admittime'].max()}")
print(f"\nFeatures: {len(df.columns)}")
print(f"Target: readmitted_30d")
print(f"Overall readmission rate: {df['readmitted_30d'].mean():.2%}")

print("\n" + "-"*60)
print("TRAIN SET")
print("-"*60)
print(f"Admissions: {len(train_df):,}")
print(f"Patients: {train_df['subject_id'].nunique():,}")
print(f"Readmission rate: {train_df['readmitted_30d'].mean():.2%}")
print(f"Date range: {train_df['admittime'].min()} to {train_df['admittime'].max()}")

print("\n" + "-"*60)
print("TEST SET")
print("-"*60)
print(f"Admissions: {len(test_df):,}")
print(f"Patients: {test_df['subject_id'].nunique():,}")
print(f"Readmission rate: {test_df['readmitted_30d'].mean():.2%}")
print(f"Date range: {test_df['admittime'].min()} to {test_df['admittime'].max()}")

print("\n" + "-"*60)
print("DATA QUALITY")
print("-"*60)
print(f"Patient overlap (train/test): {len(set(train_df['subject_id']) & set(test_df['subject_id']))} ({2.03:.2f}%)")
print(f"Missing values: 0 (all imputed)")
print(f"Temporal split: TRUE (no future information leakage)")
print("="*60)

## What we did till now:

1. We loaded raw data = 406K admissions
2. Performed temporal split = 80/20 by date (so that no future leakage)
3. Verified the patient separation = Only 2% overlap
4. Handled missing values:

- Dropped target leakage features
- Imputed labs with median (clinical standard)
- Filled categorical with mode/unknown

5. Saved clean datasets which are now ready for modeling.